# 06. Huấn luyện và so sánh mô hình

Huấn luyện Dummy, Logistic Regression, Decision Tree, Random Forest, XGBoost và CatBoost sau Feature Selection của notebook 05. Cùng một danh sách biến theo fold được dùng cho các thuật toán; các bước điền thiếu, mã hóa và scaling được học trong phần train của fold.

Đợt thực nghiệm gồm 21 cấu hình × ba fold = **63 lượt CV**, đã chạy đủ. Mỗi fold dùng **181 biến**. Danh sách **182 biến** fit trên toàn train được dành cho lần huấn luyện cuối; không dùng danh sách đó để đánh giá lại các fold.

Dữ liệu gốc trong MySQL vẫn có 199 cột. Đây là nguồn trước lọc; mô hình nhận các cột đã chọn cho từng fold. Notebook kết thúc ở so sánh CV, chưa fit mô hình cuối, calibration hay chọn ngưỡng cho bộ 182 biến.

Phương pháp Filter được trình bày trong notebook 05.

## Mục lục

- [01. Chuẩn bị dữ liệu từ MySQL](#part-1)
- [02. Áp dụng Feature Selection trong từng fold](#part-2)
- [03. Chuẩn bị pipeline và cấu hình mô hình](#part-3)
- [04. Huấn luyện và kiểm định chéo](#part-4)
- [05. So sánh và chọn ứng viên](#part-5)
- [06. Kết luận và bàn giao](#part-6)

<a id="part-1"></a>

## 01. Chuẩn bị dữ liệu từ MySQL

Đọc đúng hồ sơ train và ba fold đã chốt. Bảng MySQL giữ dữ liệu trước Feature Selection để selector có thể học riêng trong từng fold. TARGET được đọc riêng; ID, vai trò phân tập và số fold không phải đặc trưng dự báo.

In [28]:
from pathlib import Path
import sys
import json

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from sklearn.model_selection import ParameterGrid

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from connectors.Connector import Connector
from models.HomeCreditModelData import HomeCreditModelData
from models.HomeCreditTraining import HomeCreditTraining
from utils.FileUtil import FileUtil

FEATURES = ROOT / 'data/features'
DATA_VERSION = 'home-credit-v1'
feature_index = pd.read_parquet(FEATURES / 'feature_index.parquet')
pd.set_option('display.max_columns', 16)
pd.set_option('display.max_rows', 25)
display(feature_index.groupby('split').size().rename('n').to_frame())

,n
split,
application_test,48744
test,61503
train,184506
validation,61502


In [29]:
connector = Connector()
connector.connect()
server_version = connector.queryDataset('SELECT VERSION() AS version').iloc[0]['version']
database = connector.database
if database not in {'home_credit_ml', 'home_credit_submission'}:
    raise ValueError('Cần dùng CSDL nghiên cứu hoặc bản khôi phục home_credit_submission đã kiểm chứng.')
source_schema = pq.ParquetFile(FEATURES / 'b/train.parquet').schema_arrow
model_data = HomeCreditModelData(connector, source_schema)
model_data.createTables(ROOT / 'sql/01_create_schema.sql', data_version=DATA_VERSION)
print('MySQL:', server_version, '| Database:', database, '| Cột nguồn trước lọc:', len(source_schema) - 1)

MySQL: 8.0.32 | Database: home_credit_ml | Cột nguồn trước lọc: 199


In [ ]:
import_checks = model_data.importFeatures(FEATURES, feature_index)
display(import_checks)
feature_sync_checks = model_data.validateFeatures(FEATURES)
display(feature_sync_checks)


train : 184506 hồ sơ, thêm mới 0
validation : 61502 hồ sơ, thêm mới 0
test : 61503 hồ sơ, thêm mới 0
application_test : 48744 hồ sơ, thêm mới 0


split,verified_rows,verified_features,values_match
train,184506,199,True
validation,61502,199,True
test,61503,199,True
application_test,48744,199,True


In [31]:
sql_index = model_data.readSplitIndex()
sql_index = sql_index.astype(feature_index.dtypes.to_dict())
if not sql_index.equals(feature_index.reset_index(drop=True)):
    raise ValueError('ID hoặc fold MySQL khác bản đã chốt ở notebook 03.')
display(sql_index.head())

,SK_ID_CURR,split,cv_fold
0,100001,application_test,-1
1,100002,train,2
2,100003,train,2
3,100004,train,0
4,100005,application_test,-1


In [32]:
connector.disConnect()

**Kết luận:** chỉ mục phân tập đã được đối chiếu với bản đã chốt. Giá trị MySQL được so với Parquet theo lô cho cả bốn phân tập trước khi chạy CV. Nếu Feature Engineering thay đổi nhưng MySQL chưa được nạp lại, notebook dừng để đối soát thay vì train trên dữ liệu cũ.


<a id="part-2"></a>

## 02. Áp dụng Feature Selection trong từng fold

Áp dụng cùng hàm `select_filter_features` và các tham số đã chốt trong notebook 05: loại cột hằng và biến số tương quan Pearson cao. Trong mỗi lượt CV, selector chỉ nhìn hai phần dùng để fit; phần giữ lại không tham gia chọn biến. Selector không nhận TARGET.

Fit lại selector trên toàn train chỉ để xuất danh sách cho mô hình cuối. Hai loại danh sách này có phạm vi sử dụng khác nhau và được lưu riêng.

In [33]:
from models.FeatureSelection import select_filter_features

FILTER_REPORTS = ROOT / 'reports/modeling/feature_selection_redo_20261029'
FILTER_REPORTS.mkdir(parents=True, exist_ok=True)
FILTER_CACHE = FILTER_REPORTS / 'cache'

selection_connector = Connector()
selection_connector.connect()
try:
    selection_reader = HomeCreditModelData(
        selection_connector, pq.ParquetFile(FEATURES / 'b/train.parquet').schema_arrow)
    selection_data = selection_reader.readFeatures('train')
    selection_target = selection_reader.readLabels('train')
    selection_split = selection_reader.readSplitIndex().set_index('SK_ID_CURR')
finally:
    selection_connector.disConnect()

selection_folds = selection_split.loc[selection_data.index, 'cv_fold'].astype('int8')
selection_schema = FileUtil.readJson(ROOT / 'configs/preprocessing_schema.json')['schemas']['B']
selection_data = selection_data.loc[:, selection_schema['feature_order']]
if not selection_data.index.equals(selection_target.index):
    raise ValueError('Đặc trưng và nhãn train không cùng ID/thứ tự.')
if set(selection_folds.unique()) != {0, 1, 2}:
    raise ValueError('Không khớp ba CV fold đã chốt.')

selection_policy = {
    'method': 'constant filter and Pearson redundancy filter',
    'correlation_limit': 0.90,
    'minimum_pairwise_overlap': 0.50,
    'representative_rule': 'lower missingness, then original schema order',
    'target_used_by_selector': False,
}
selected_by_fold = {}
dropped_by_fold = []
selection_rows = []
for fold in [0, 1, 2]:
    fit_mask = selection_folds.ne(fold)
    selected, dropped = select_filter_features(
        selection_data.loc[fit_mask], selection_schema,
        correlation_limit=selection_policy['correlation_limit'],
        min_pairwise_overlap=selection_policy['minimum_pairwise_overlap'])
    selected_by_fold[str(fold)] = selected
    dropped_by_fold.extend({'fold': fold, **row} for row in dropped)
    selection_rows.append({
        'fold': fold, 'fit_n': int(fit_mask.sum()),
        'probe_n': int((~fit_mask).sum()),
        'input_features': len(selection_schema['feature_order']),
        'selected_features': len(selected),
        'removed_features': len(dropped),
    })

final_selected, final_dropped = select_filter_features(
    selection_data, selection_schema,
    correlation_limit=selection_policy['correlation_limit'],
    min_pairwise_overlap=selection_policy['minimum_pairwise_overlap'])
final_schema = {
    'feature_order': final_selected,
    'numeric': [column for column in selection_schema['numeric'] if column in final_selected],
    'categorical': [column for column in selection_schema['categorical'] if column in final_selected],
}
pd.DataFrame(selection_rows).to_csv(FILTER_REPORTS / 'selection_summary_by_fold.csv', index=False)
pd.DataFrame(dropped_by_fold).to_csv(FILTER_REPORTS / 'dropped_features_by_fold.csv', index=False)
pd.DataFrame(final_dropped).to_csv(FILTER_REPORTS / 'final_dropped_features.csv', index=False)
(FILTER_REPORTS / 'selected_features_by_fold.json').write_text(
    json.dumps(selected_by_fold, indent=2), encoding='utf-8')
(FILTER_REPORTS / 'final_selected_features_train_only.json').write_text(
    json.dumps(final_selected, indent=2), encoding='utf-8')
(FILTER_REPORTS / 'selected_schema_train_only.json').write_text(
    json.dumps(final_schema, indent=2), encoding='utf-8')
display(pd.DataFrame(selection_rows))
print('Fit toàn train:', len(final_selected), 'biến; dùng riêng cho lần fit cuối.')


,fold,fit_n,probe_n,input_features,selected_features,removed_features
0,0,123004,61502,199,181,18
1,1,123004,61502,199,181,18
2,2,123004,61502,199,181,18


Fit toàn train: 182 biến; dùng riêng cho lần fit cuối.


**Kết luận:** mỗi fold giữ 181/199 biến; toàn train giữ 182/199 biến. Các mô hình bên dưới nhận danh sách của đúng fold qua `feature_columns_by_fold`. Dữ liệu trước lọc có 199 cột không có nghĩa là mô hình được fit trên cả 199 cột.

<a id="part-3"></a>

## 03. Chuẩn bị pipeline và cấu hình mô hình

Chốt cấu hình trước vòng CV. Dummy tạo mốc đối chứng; LR là mô hình tuyến tính; DT, RF, XGBoost và CatBoost bổ sung các cách học quan hệ phi tuyến. AP trung bình ngoài fold là tiêu chí chính vì lớp khó khăn thanh toán chiếm tỷ lệ thấp.

LR, DT và RF dùng pipeline của `HomeCreditTraining`: điền thiếu, tạo cờ thiếu và mã hóa trong phần fit. LR so sánh nhánh standard với log các cột tiền còn được giữ sau lọc. XGBoost và CatBoost dùng cách xử lý đặc trưng riêng đã chốt, cũng chỉ học từ phần fit.

In [ ]:
# Các lựa chọn thực nghiệm giữ nguyên; hiển thị trực tiếp để dễ giải thích.
baseline_plan = {'seed': 42,
 'log_columns': ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE'],
 'LR': {'C': [0.1, 1.0],
        'class_weight': [None, 'balanced'],
        'preprocessing': ['standard', 'log_amounts'],
        'solver': 'liblinear',
        'l1_ratio': 0.0,
        'max_iter': 1000,
        'tol': 0.0001},
 'DT': {'max_depth': [6, 10],
        'min_samples_leaf': 100,
        'class_weight': [None, 'balanced'],
        'criterion': 'gini'},
 'RF': {'n_estimators': 100,
        'max_depth': [10, 16],
        'min_samples_leaf': 30,
        'max_features': 'sqrt',
        'class_weight': [None, 'balanced'],
        'n_jobs': 1,
        'criterion': 'gini'}}


In [ ]:
lr_settings = baseline_plan['LR']
tree_settings = baseline_plan['DT']
forest_settings = baseline_plan['RF']
grids = {
    'Dummy': [{'strategy': 'prior'}],
    'LR': list(ParameterGrid({
        'C': lr_settings['C'], 'class_weight': lr_settings['class_weight'],
        'preprocessing': lr_settings['preprocessing']
    })),
    'DT': list(ParameterGrid({
        'max_depth': tree_settings['max_depth'], 'class_weight': tree_settings['class_weight']
    })),
    'RF': list(ParameterGrid({
        'max_depth': forest_settings['max_depth'], 'class_weight': forest_settings['class_weight']
    }))
}

In [ ]:
from importlib.metadata import version

boosting_plan = {
    'seed': 42,
    'threads': 4,
    'libraries': {'XGB': version('xgboost'), 'CAT': version('catboost')},
    'XGB': {
        'objective': 'binary:logistic', 'eval_metric': 'logloss',
        'tree_method': 'hist', 'enable_categorical': True,
        'n_estimators': 500, 'learning_rate': 0.05, 'max_bin': 128,
        'min_child_weight': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
        'reg_lambda': 5.0, 'reg_alpha': 0.0, 'scale_pos_weight': 1.0
    },
    'CAT': {
        'loss_function': 'Logloss', 'iterations': 500, 'learning_rate': 0.05,
        'l2_leaf_reg': 5.0, 'boosting_type': 'Plain',
        'bootstrap_type': 'Bernoulli', 'subsample': 0.8, 'rsm': 0.8,
        'max_ctr_complexity': 1, 'used_ram_limit': '1gb',
        'allow_writing_files': False, 'verbose': False
    }
}
boosting_grids = {
    'XGB': [{'max_depth': 3}, {'max_depth': 5}],
    'CAT': [{'depth': 4}, {'depth': 6}]
}

In [ ]:
budget = pd.DataFrame([
    {'model': model, 'configs': len(configs), 'cv_fits': len(configs) * 3}
    for model, configs in {**grids, **boosting_grids}.items()
])
display(budget)
print('Tổng cấu hình:', budget['configs'].sum(), '| Tổng lượt CV:', budget['cv_fits'].sum())

,model,configs,cv_fits
0,Dummy,1,3
1,LR,8,24
2,DT,4,12
3,RF,4,12
4,XGB,2,6
5,CAT,2,6


Tổng cấu hình: 21 | Tổng lượt CV: 63


**Kết luận:** có 21 cấu hình, tương ứng 63 lượt CV. RF dùng một luồng và boosting dùng bốn luồng, khớp điều kiện chạy của đợt sau Feature Selection. Mọi họ mô hình dùng cùng ID và danh sách biến theo fold.

<a id="part-4"></a>

## 04. Huấn luyện và kiểm định chéo

Mỗi cấu hình học trên hai fold rồi đánh giá trên fold còn lại, lặp ba lần. Ghi AP, ROC-AUC, Brier, AP trên phần fit và thời gian. Điểm trên phần fit dùng xem mức quá khớp; việc so sánh dùng điểm ngoài fold.

Đợt chạy được lưu trong `reports/modeling/feature_selection_redo_20261029`. Khi chạy lại, code kiểm tra và đọc checkpoint đúng dữ liệu, cấu hình và danh sách biến; chỉ fit khi chưa có checkpoint hợp lệ. Checkpoint của đợt 28/10 không được dùng cho đợt này; khi chạy lại có thể dùng checkpoint hợp lệ của chính đợt 29/10. Validation và test không tham gia vòng này.

In [ ]:
filtered_schema = {
    **selection_schema,
    'selection_policy': selection_policy,
    'selected_features_by_fold': selected_by_fold,
}

filtered_baseline_plan = {**baseline_plan, 'feature_selection': selection_policy}
filtered_baseline_schemas = {'B_FILTER': filtered_schema}
filtered_baseline_key = HomeCreditTraining.makeCacheKey(
    selection_data, selection_target, selection_folds,
    filtered_baseline_plan, filtered_baseline_schemas,
    'feature-selection-redo-20261029', rf_jobs=1)
filtered_baseline_training = HomeCreditTraining(
    filtered_baseline_plan, filtered_baseline_schemas,
    FILTER_CACHE / 'baseline' / filtered_baseline_key,
    filtered_baseline_key, rf_jobs=1)

filtered_baseline_records = []
for model_name, configurations in grids.items():
    for number, configuration in enumerate(configurations, 1):
        case_id = f'B_FILTER_{model_name}_{number:02d}'
        case = {'set': 'B_FILTER', 'model': model_name, 'config': configuration}
        filtered_baseline_records.extend(filtered_baseline_training.runCrossValidation(
            case_id, case, selection_data, selection_target, selection_folds,
            feature_columns_by_fold=selected_by_fold))

filtered_boosting_plan = {**boosting_plan, 'feature_selection': selection_policy}
filtered_boosting_schemas = {'B_FILTER': filtered_schema}
filtered_boosting_key = HomeCreditTraining.makeCacheKey(
    selection_data, selection_target, selection_folds,
    filtered_boosting_plan, filtered_boosting_schemas,
    'feature-selection-boosting-redo-20261029', rf_jobs=boosting_plan['threads'])
filtered_boosting_training = HomeCreditTraining(
    filtered_boosting_plan, filtered_boosting_schemas,
    FILTER_CACHE / 'boosting' / filtered_boosting_key,
    filtered_boosting_key)

filtered_boosting_cases = {}
for model_name, configurations in boosting_grids.items():
    for number, configuration in enumerate(configurations, 1):
        case_id = f'B_FILTER_{model_name}_{number:02d}'
        filtered_boosting_cases[case_id] = {
            'set': 'B_FILTER', 'model': model_name, 'config': configuration}

filtered_boosting_records = []
for case_id, case in filtered_boosting_cases.items():
    filtered_boosting_records.extend(filtered_boosting_training.runBoostingCrossValidation(
        case_id, case, selection_data, selection_target, selection_folds,
        feature_columns_by_fold=selected_by_fold))

filtered_cv = pd.DataFrame(filtered_baseline_records + filtered_boosting_records)
expected_filtered_fits = (sum(map(len, grids.values())) +
                          len(filtered_boosting_cases)) * 3
if len(filtered_cv) != expected_filtered_fits or filtered_cv.duplicated(['case_id', 'fold']).any():
    raise ValueError('Kết quả Feature Selection CV chưa đủ hoặc bị trùng case/fold.')
expected_features_by_fold = {int(fold): len(columns)
                             for fold, columns in selected_by_fold.items()}
if not filtered_cv['features_n'].eq(filtered_cv['fold'].map(expected_features_by_fold)).all():
    raise ValueError('Số biến ở các fold không khớp danh sách selector của fold.')

filtered_cv.to_csv(FILTER_REPORTS / 'selected_cv_folds.csv', index=False)
(FILTER_REPORTS / 'execution.json').write_text(json.dumps({
    'status': 'cv_complete', 'cv_fits': len(filtered_cv),
    'expected_cv_fits': expected_filtered_fits,
    'baseline_cache_key': filtered_baseline_key,
    'boosting_cache_key': filtered_boosting_key,
    'previous_experiment_checkpoints_reused': False,
    'same_experiment_checkpoint_reuse_allowed': True,
    'validation_labels_used': False,
    'test_labels_used': False,
    'final_model_or_dashboard_changed': False,
}, ensure_ascii=False, indent=2), encoding='utf-8')
print('Hoàn tất', len(filtered_cv), 'lượt CV mới trên bộ B đã lọc.')


B_FILTER_Dummy_01 fold 0 : dùng checkpoint; fit_s bên dưới là thời gian fit đã lưu
B_FILTER_Dummy_01 fold 0 AP= 0.08073 fit_s= 5.1 converged= True
B_FILTER_Dummy_01 fold 1 : dùng checkpoint; fit_s bên dưới là thời gian fit đã lưu
B_FILTER_Dummy_01 fold 1 AP= 0.08073 fit_s= 3.9 converged= True
B_FILTER_Dummy_01 fold 2 : dùng checkpoint; fit_s bên dưới là thời gian fit đã lưu
B_FILTER_Dummy_01 fold 2 AP= 0.08073 fit_s= 3.9 converged= True
B_FILTER_LR_01 fold 0 : dùng checkpoint; fit_s bên dưới là thời gian fit đã lưu
B_FILTER_LR_01 fold 0 AP= 0.24921 fit_s= 36.8 converged= True
B_FILTER_LR_01 fold 1 : dùng checkpoint; fit_s bên dưới là thời gian fit đã lưu
B_FILTER_LR_01 fold 1 AP= 0.25172 fit_s= 39.2 converged= True
B_FILTER_LR_01 fold 2 : dùng checkpoint; fit_s bên dưới là thời gian fit đã lưu
B_FILTER_LR_01 fold 2 AP= 0.25172 fit_s= 43.9 converged= True
B_FILTER_LR_02 fold 0 : dùng checkpoint; fit_s bên dưới là thời gian fit đã lưu
B_FILTER_LR_02 fold 0 AP= 0.25084 fit_s= 52.3 converg

**Kết luận:** đã hoàn tất 63 lượt, mỗi cấu hình có đủ ba fold và mỗi fold dùng 181 biến. Kết quả từng lượt nằm trong `selected_cv_folds.csv`; checkpoint lưu dự báo và thông tin để đối soát khi chạy lại.

<a id="part-5"></a>

## 05. So sánh và chọn ứng viên

Tổng hợp AP trung bình và độ lệch chuẩn giữa các fold; đọc thêm ROC-AUC, Brier và chênh lệch AP giữa phần fit với phần đánh giá. Chọn cấu hình tốt nhất của từng họ theo AP trung bình, sau đó đối chiếu các họ trên cùng dữ liệu đã lọc.

In [ ]:
selected_summary = filtered_cv.groupby(['model', 'config'], as_index=False).agg(
    case_id=('case_id', 'first'), folds_n=('fold', 'size'),
    features_per_fold=('features_n', 'min'),
    mean_ap=('ap', 'mean'), std_ap=('ap', 'std'),
    mean_roc_auc=('roc_auc', 'mean'), mean_brier=('brier', 'mean'),
    mean_train_ap=('train_ap', 'mean'))
selected_summary['train_cv_ap_gap'] = selected_summary['mean_train_ap'] - selected_summary['mean_ap']
selected_summary = selected_summary.sort_values(
    ['mean_ap', 'mean_roc_auc'], ascending=False).reset_index(drop=True)
best_selected = selected_summary.groupby('model', sort=False).head(1)

selected_summary.to_csv(FILTER_REPORTS / 'selected_cv_summary.csv', index=False)
best_selected.to_csv(FILTER_REPORTS / 'selected_best_by_family.csv', index=False)
display(best_selected[['model', 'config', 'features_per_fold', 'mean_ap', 'std_ap', 'mean_roc_auc',
                       'mean_brier', 'train_cv_ap_gap']])

,model,config,features_per_fold,mean_ap,std_ap,mean_roc_auc,mean_brier,train_cv_ap_gap
0,CAT,"{""depth"": 6}",181,0.267634,0.001476,0.776217,0.066621,0.099931
1,XGB,"{""max_depth"": 5}",181,0.266853,0.003741,0.776353,0.066670,0.221903
4,LR,"{""C"": 0.1, ""class_weight"": null, ""preprocessing"": ""log_amounts""}",181,0.252025,0.001026,0.767299,0.067332,0.008911
12,RF,"{""class_weight"": null, ""max_depth"": 16}",181,0.243669,0.000888,0.757852,0.069040,0.272693
16,DT,"{""class_weight"": ""balanced"", ""max_depth"": 10}",181,0.182422,0.001711,0.699892,0.205076,0.072966
20,Dummy,"{""strategy"": ""prior""}",181,0.080729,0.000000,0.500000,0.074212,0.000000


**Kết luận:** CatBoost depth=6 có AP trung bình cao nhất (0,267634), tiếp theo là XGBoost max_depth=5 (0,266853). ROC-AUC tương ứng là 0,776217 và 0,776353. CatBoost có độ lệch chuẩn AP thấp hơn XGBoost trong ba fold này; chênh lệch điểm nhỏ chưa đủ để khẳng định mô hình nào luôn tốt hơn trên dữ liệu mới.

LR đạt AP 0,252025 và chênh AP train–CV 0,008911, thấp hơn các mô hình cây. RF đạt AP 0,243669 nhưng chênh train–CV 0,272693, cho thấy dấu hiệu quá khớp. DT đạt 0,182422; Dummy đạt 0,080729.

<a id="part-6"></a>

## 06. Kết luận và bàn giao

Vòng so sánh CV sau Feature Selection đã hoàn tất. CatBoost depth=6 là ứng viên ưu tiên theo AP; LR được giữ làm đối chứng đơn giản. Đây là lựa chọn trong phạm vi cấu hình đã thử, chưa phải đánh giá hiệu quả vận hành tại một ngưỡng cảnh báo.

Notebook 05 đã chốt phương pháp và danh sách biến; notebook 06 đã hoàn tất huấn luyện và so sánh bằng train CV. Notebook 07 đã tiếp tục quy trình: fit model cuối với 182 biến, đánh giá calibration và chọn ngưỡng trên validation. Notebook 08 đã giải thích và minh họa sử dụng artifact đó. Artifact đang triển khai chưa được thay trong notebook này.
